# 🚀 Path 3: Image-to-Class
**Model**: 2D Image Classifier

### Configuration:
- **Dataset**: MIT-BIH Arrhythmia (10% Sample for Testing)
- **Epochs**: 2
- **Environment**: Production-Ready Torch Pipeline

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import os
from torch.utils.data import DataLoader, TensorDataset, Dataset

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_PATH = '../mitbih_train.csv'

def load_production_data(sample_frac=0.1):
    if not os.path.exists(DATA_PATH): return None
    df = pd.read_csv(DATA_PATH, header=None)
    return df.sample(frac=sample_frac, random_state=42)

df_sample = load_production_data(0.1)
print(f'📦 Loaded {len(df_sample)} rows (10% of dataset)')

In [ ]:
import cv2
def fast_signal_to_image(signal, img_size=(128, 128)):
    img = np.zeros((img_size[0], img_size[1]), dtype=np.float32)
    x_coords = np.linspace(0, img_size[1]-1, len(signal)).astype(int)
    y_coords = ((1 - signal) * (img_size[0]-1)).astype(int)
    for i in range(len(x_coords)-1):
        cv2.line(img, (x_coords[i], y_coords[i]), (x_coords[i+1], y_coords[i+1]), 1.0, 1)
    return img

class ImageClassifier2D(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 16 * 16, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )
    def forward(self, x): return self.classifier(self.features(x))

model = ImageClassifier2D().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print('🎨 Generating images for 10% sample...')
X_imgs = torch.tensor([fast_signal_to_image(s) for s in df_sample.iloc[:200, :187].values], dtype=torch.float32).unsqueeze(1).to(DEVICE)
y_labels = torch.tensor(df_sample.iloc[:200, 187].values, dtype=torch.long).to(DEVICE)

print('🚀 Production Training: Image-to-Class Visual Diagnosis')
for epoch in range(2):
    optimizer.zero_grad()
    outputs = model(X_imgs)
    loss = criterion(outputs, y_labels)
    loss.backward()
    optimizer.step()
    acc = (outputs.argmax(1) == y_labels).float().mean()
    print(f'Epoch {epoch+1}: Loss={loss.item():.4f}, Accuracy={acc:.4f}')

if not os.path.exists('models'): os.makedirs('models')
torch.save(model.state_dict(), 'models/image_to_class.pth')
print('✅ Model Saved.')